# Lecture 09 — Inferencia con Modelos Preentrenados: YOLO

**Introduccion a la Inteligencia Artificial — EAFIT 2026-01**

---

En esta clase vamos a trabajar con **YOLO** (*You Only Look Once*), uno de los modelos mas populares y eficientes para **deteccion de objetos** en imagenes y video.

Veremos:
1. Que es YOLO y como funciona a nivel conceptual.
2. Como instalar y cargar un modelo preentrenado con la libreria `ultralytics`.
3. Como ejecutar inferencia sobre una imagen.
4. Como explorar los resultados (bounding boxes, clases, probabilidades de confianza).
5. Como visualizar las detecciones.
6. Como ajustar el umbral de confianza.
7. Como probar con tus propias imagenes.


## 1. ¿Que es YOLO?

YOLO es una **red neuronal convolucional** disenada para detectar objetos en imagenes en tiempo real.

### ¿Como funciona?

A diferencia de los clasificadores de imagenes (que dicen *que* hay en toda la imagen), YOLO resuelve dos problemas al mismo tiempo:
1. **Localizacion**: *¿Donde* estan los objetos? (coordenadas de la caja delimitadora o *bounding box*).
2. **Clasificacion**: *¿Que* es cada objeto? (etiqueta de clase y probabilidad).

La clave de YOLO es que procesa la **imagen completa en una sola pasada** (*You Only Look Once*), dividiendola en una grilla y prediciendo bounding boxes y probabilidades simultaneamente. Esto lo hace extremadamente rapido, capaz de procesar video en tiempo real a mas de 30 FPS.

### El dataset COCO

Los modelos preentrenados de YOLO que usaremos fueron entrenados sobre **COCO** (*Common Objects in Context*), un dataset estandar con **80 clases** de objetos comunes: personas, carros, perros, gatos, sillas, botellas, celulares, etc.


## 2. Instalacion

Solo necesitamos una libreria: **`ultralytics`**, que incluye YOLO y todas sus dependencias.

In [ ]:
%pip install -q -U ultralytics


## 3. Cargando el modelo

Al ejecutar `YOLO("yolo26n.pt")` por primera vez, el modelo se **descarga automaticamente** desde los servidores de Ultralytics (pesa solo ~6 MB para la version nano).

Las versiones de YOLO vienen en diferentes tamanos:
- **nano (n)**: el mas rapido y ligero (~6 MB), ideal para CPU y dispositivos embebidos.
- **small (s)**: balance entre velocidad y precision (~22 MB).
- **medium (m)**, **large (l)**, **xlarge (x)**: progresivamente mas precisos pero mas lentos y pesados.

Para esta sesion usamos **nano**, que corre perfectamente en CPU sin necesidad de GPU.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

print(f"Modelo cargado: {type(model)}")
print(f"Numero de clases que puede detectar: {len(model.names)}")
print(f"\nPrimeras 5 clases: {list(model.names.values())[:5]}")

In [ ]:
print("Todas las clases del dataset COCO:\n")
for id, nombre in model.names.items():
    print(f"  {id:3d}: {nombre}")

## 4. Inferencia sobre una imagen

Vamos a descargar una imagen de prueba y ejecutar YOLO sobre ella.

YOLO acepta como entrada:
- Una ruta a un archivo local (ej. `"mi_imagen.jpg"`)
- Una URL de internet
- Un array de NumPy (imagen cargada con OpenCV o PIL)
- Un video o stream de camara


In [ ]:
import urllib.request
from pathlib import Path

# Descargar imagen de prueba (imagen de ejemplo de Ultralytics)
url = "https://ultralytics.com/images/bus.jpg"
imagen_path = "images/bus.jpg"

Path(imagen_path).parent.mkdir(parents=True, exist_ok=True)  # crea la carpeta images/ si no existe

if not Path(imagen_path).exists():
    urllib.request.urlretrieve(url, imagen_path) #Descargar la imagen desde la URL y guardarla localmente
    print(f"Imagen descargada: {imagen_path}")
else:
    print(f"Imagen ya existe: {imagen_path}")

# Mostrar imagen original
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open(imagen_path)
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.title("Imagen original")
plt.axis("off")
plt.show()
print(f"Tamano de la imagen: {img.size[0]}x{img.size[1]} pixeles")

In [ ]:
resultados = model(imagen_path)

print(f"Tipo del resultado: {type(resultados)}")
print(f"Numero de imagenes procesadas: {len(resultados)}")
#print(f"Resultados: {resultados[0]}")

## 5. Explorando el objeto de resultados

YOLO retorna una lista de objetos `Results`. Cada uno contiene toda la informacion de las detecciones para esa imagen:

- **`boxes`**: las cajas delimitadoras detectadas.
  - `boxes.xyxy`: coordenadas en pixeles `[x1, y1, x2, y2]` (esquina superior-izquierda, esquina inferior-derecha).
  - `boxes.conf`: nivel de confianza de la prediccion (entre 0.0 y 1.0).
  - `boxes.cls`: id numerico de la clase detectada.
- **`names`**: diccionario mapeando id de clase a nombre legible (ej. `{0: 'person', 16: 'dog', ...}`).


In [ ]:
r = resultados[0]

print("=== Informacion general ===")
print(f"Imagen original: {r.path}")
print(f"Tamano procesado: {r.orig_shape}")
print(f"Objetos detectados: {len(r.boxes)}")


print("\n=== Detecciones ===")
for i, box in enumerate(r.boxes):
    clase_id  = int(box.cls[0])
    clase_nom = model.names[clase_id]
    confianza = float(box.conf[0])
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    
    print(f"  [{i+1}] {clase_nom:15s} | confianza: {confianza:.2f} | bbox: ({x1:.0f},{y1:.0f}) → ({x2:.0f},{y2:.0f})")

## 6. Visualizando las detecciones

YOLO puede generar automaticamente la imagen anotada con los bounding boxes y etiquetas.

### Conversion BGR a RGB

`r.plot()` retorna la imagen usando **OpenCV** internamente, que historicamente guarda los canales de color en orden **BGR** (Azul, Verde, Rojo). Matplotlib en cambio espera orden **RGB** (Rojo, Verde, Azul).

```
OpenCV / YOLO         Matplotlib
canal 0 = Azul   →   canal 0 = Rojo
canal 1 = Verde  →   canal 1 = Verde
canal 2 = Rojo   →   canal 2 = Azul
```

Si mostramos la imagen sin convertir, los colores quedan **invertidos** (lo rojo aparece azul y viceversa).

La conversion se hace con slicing de NumPy:

```python
imagen_anotada_rgb = imagen_anotada[:, :, ::-1]
#                                   │    │   └── invierte el orden de los 3 canales
#                                   │    └────── todos los pixeles del ancho
#                                   └──────────── todos los pixeles del alto
```

El `::-1` recorre el eje de canales al reves: `[0

In [ ]:
import numpy as np

# r.plot() retorna la imagen anotada como array NumPy (BGR → convertir a RGB para matplotlib)
imagen_anotada = r.plot()
imagen_anotada_rgb = imagen_anotada[:, :, ::-1]  # BGR a RGB

plt.figure(figsize=(12, 7))
plt.imshow(imagen_anotada_rgb)
plt.title(f"YOLO — {len(r.boxes)} objetos detectados")
plt.axis("off")
plt.show()

## 7. Ajustando el umbral de confianza

El parametro `conf` controla que tan "seguro" debe estar YOLO para reportar una deteccion.

- `conf=0.25` → reporta detecciones con al menos 25% de confianza (mas detecciones, posible ruido / falsos positivos).
- `conf=0.50` → valor tipico para uso general.
- `conf=0.75` → solo detecciones muy seguras (menos falsos positivos, pero podria ignorar objetos reales).

Veamos como cambia el resultado con tres umbrales diferentes sobre la misma imagen:


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
umbrales = [0.25, 0.50, 0.75]

for ax, umbral in zip(axes, umbrales): #Iterar sobre cada subplot y umbral de confianza
    r = model(imagen_path, conf=umbral, verbose=False)[0] #Ejecutar inferencia con el umbral de confianza especificado y obtener el resultado para la primera imagen
    img_anotada = r.plot()[:, :, ::-1] 
    ax.imshow(img_anotada)
    ax.set_title(f"conf={umbral} → {len(r.boxes)} detecciones")
    ax.axis("off")

plt.suptitle("Efecto del umbral de confianza", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Ejercicio — Prueba con tu propia imagen

Sube cualquier imagen y ejecuta YOLO sobre ella. Puedes usar una foto tuya, de tu escritorio, o descargar cualquier imagen de internet.

In [ ]:
## EJERCICIO: cambia este path por el de tu propia imagen
MI_IMAGEN = "xxxxxx.jpg"   # <-- reemplaza con tu archivo, ej: "mi_foto.jpg"

r = model(MI_IMAGEN, conf="???", verbose=False)[0] # Reemplaza "???" por el umbral de confianza que quieras probar, ej: 0.25, 0.50, 0.75
img_anotada = r.plot()[:, :, ::-1]

plt.figure(figsize=(12, 7))
plt.imshow(img_anotada)
plt.title(f"Tu imagen — {len(r.boxes)} objetos detectados")
plt.axis("off")
plt.show()

print("\nDetecciones:")
for box in r.boxes:
    clase = model.names[int(box.cls[0])]
    conf  = float(box.conf[0])
    print(f"  - {clase} ({conf:.0%} de confianza)")